##

In [3]:
import pandas as pd
import folium
import datetime
from abc import ABC, abstractmethod

# -------------------------------------------------------------------
# BAGIAN 1: DEFINISI KELAS (termasuk kelas baru hasil penugasan)
# -------------------------------------------------------------------

class Lokasi(ABC):
    """Kelas abstrak dasar untuk merepresentasikan sebuah lokasi geografis."""

    def __init__(self, nama: str, latitude: float, longitude: float):
        self.nama = str(nama) if nama else "Tanpa Nama"
        try:
            self.latitude = float(latitude)
            self.longitude = float(longitude)
        except (ValueError, TypeError):
            self.latitude = 0.0
            self.longitude = 0.0

    def get_koordinat(self) -> tuple:
        """Mengembalikan koordinat lokasi sebagai tuple (latitude, longitude)."""
        return (self.latitude, self.longitude)

    @abstractmethod
    def get_info_popup(self) -> str:
        """Mengembalikan string HTML untuk konten popup marker peta."""
        pass

    def __repr__(self) -> str:
        return (f"{type(self).__name__}(nama='{self.nama}', "
                f"lat={self.latitude:.4f}, lon={self.longitude:.4f})")

    def __str__(self) -> str:
        return f"{self.nama} [{type(self).__name__}]"


class TempatWisata(Lokasi):
    """Merepresentasikan tempat wisata (alam, sejarah, landmark)."""

    def __init__(self, nama: str, latitude: float, longitude: float,
                 jenis: str, deskripsi: str):
        super().__init__(nama, latitude, longitude)
        self.jenis_wisata = str(jenis) if jenis else "Umum"
        self.deskripsi = str(deskripsi) if deskripsi else "Tidak ada deskripsi."

    def get_info_popup(self) -> str:
        return (f"<h4><b>{self.nama}</b></h4>"
                f"<i>{self.jenis_wisata}</i><br><br>"
                f"{self.deskripsi}<br><br>"
                f"<small>Koordinat: ({self.latitude:.4f}, {self.longitude:.4f})</small>")


class Kuliner(Lokasi):
    """Merepresentasikan lokasi kuliner atau restoran."""

    def __init__(self, nama: str, latitude: float, longitude: float,
                 menu_andalan: str):
        super().__init__(nama, latitude, longitude)
        self.menu_andalan = str(menu_andalan) if menu_andalan else "Tidak diketahui"

    def get_info_popup(self) -> str:
        return (f"<h4><b>{self.nama}</b></h4>"
                f"<i>Kuliner</i><br><br>"
                f"Menu Andalan: {self.menu_andalan}<br><br>"
                f"<small>Koordinat: ({self.latitude:.4f}, {self.longitude:.4f})</small>")


class TempatIbadah(Lokasi):
    """Merepresentasikan tempat ibadah."""

    def __init__(self, nama: str, latitude: float, longitude: float,
                 agama: str = "Umum", deskripsi: str = ""):
        super().__init__(nama, latitude, longitude)
        self.agama = str(agama) if agama else "Umum"
        self.deskripsi = str(deskripsi) if deskripsi else "Tempat Ibadah"

    def get_info_popup(self) -> str:
        return (f"<h4><b>{self.nama}</b></h4>"
                f"<i>Tempat Ibadah ({self.agama})</i><br><br>"
                f"{self.deskripsi}<br><br>"
                f"<small>Koordinat: ({self.latitude:.4f}, {self.longitude:.4f})</small>")


# -----------------------------------------------------------------------
# KELAS BARU PENUGASAN 1: Museum, TamanKota, KantorPemerintahan
# -----------------------------------------------------------------------

class Museum(Lokasi):
    """
    Kelas baru (Penugasan) untuk merepresentasikan museum.
    Mewarisi dari kelas abstrak Lokasi.
    """

    def __init__(self, nama: str, latitude: float, longitude: float,
                 koleksi_utama: str = "Umum", deskripsi: str = ""):
        super().__init__(nama, latitude, longitude)
        self.koleksi_utama = str(koleksi_utama) if koleksi_utama else "Umum"
        self.deskripsi = str(deskripsi) if deskripsi else "Museum"

    def get_info_popup(self) -> str:
        return (f"<h4><b>{self.nama}</b></h4>"
                f"<i>Museum</i><br><br>"
                f"{self.deskripsi}<br><br>"
                f"Koleksi Utama: {self.koleksi_utama}<br><br>"
                f"<small>Koordinat: ({self.latitude:.4f}, {self.longitude:.4f})</small>")


class TamanKota(Lokasi):
    """
    Kelas baru (Penugasan) untuk merepresentasikan taman kota.
    Mewarisi dari kelas abstrak Lokasi.
    """

    def __init__(self, nama: str, latitude: float, longitude: float,
                 fasilitas: str = "Area hijau", deskripsi: str = ""):
        super().__init__(nama, latitude, longitude)
        self.fasilitas = str(fasilitas) if fasilitas else "Area hijau"
        self.deskripsi = str(deskripsi) if deskripsi else "Taman Kota"

    def get_info_popup(self) -> str:
        return (f"<h4><b>{self.nama}</b></h4>"
                f"<i>Taman Kota</i><br><br>"
                f"{self.deskripsi}<br><br>"
                f"Fasilitas: {self.fasilitas}<br><br>"
                f"<small>Koordinat: ({self.latitude:.4f}, {self.longitude:.4f})</small>")


class KantorPemerintahan(Lokasi):
    """
    Kelas baru (Penugasan) untuk merepresentasikan kantor pemerintahan.
    Mewarisi dari kelas abstrak Lokasi.
    """

    def __init__(self, nama: str, latitude: float, longitude: float,
                 instansi: str = "Pemerintah Daerah", deskripsi: str = ""):
        super().__init__(nama, latitude, longitude)
        self.instansi = str(instansi) if instansi else "Pemerintah Daerah"
        self.deskripsi = str(deskripsi) if deskripsi else "Kantor Pemerintahan"

    def get_info_popup(self) -> str:
        return (f"<h4><b>{self.nama}</b></h4>"
                f"<i>Kantor Pemerintahan</i><br><br>"
                f"{self.deskripsi}<br><br>"
                f"Instansi: {self.instansi}<br><br>"
                f"<small>Koordinat: ({self.latitude:.4f}, {self.longitude:.4f})</small>")


# -------------------------------------------------------------------
# FUNGSI BACA DATA CSV
# -------------------------------------------------------------------

def baca_data_lokasi(nama_file: str) -> pd.DataFrame | None:
    """Membaca data lokasi dari file CSV menggunakan Pandas."""
    try:
        dataframe = pd.read_csv(nama_file)
        return dataframe
    except FileNotFoundError:
        print(f"ERROR: File '{nama_file}' tidak ditemukan!")
        return None
    except Exception as e:
        print(f"ERROR saat membaca file CSV: {type(e).__name__} - {e}")
        return None


# -------------------------------------------------------------------
# FUNGSI BUAT OBJEK (dimodifikasi untuk mengenali tipe baru)
# -------------------------------------------------------------------

def buat_objek_lokasi_dari_df(dataframe: pd.DataFrame) -> list:
    """
    Mengiterasi DataFrame Pandas dan membuat list objek Lokasi.
    Mendukung tipe: Wisata, Landmark, Kuliner, Tempat Ibadah,
    Museum, Taman Kota, dan Kantor Pemerintahan. (Penugasan)
    """
    list_objek_lokasi = []
    if dataframe is None or dataframe.empty:
        print("DataFrame kosong atau None, tidak ada objek dibuat.")
        return list_objek_lokasi

    print("\nMembuat objek dari DataFrame...")
    for index, row in dataframe.iterrows():
        nama      = row.get('Nama', None)
        lat       = row.get('Latitude', None)
        lon       = row.get('Longitude', None)
        tipe      = row.get('Tipe', 'Lainnya')
        deskripsi = row.get('Deskripsi', '')

        if nama is None or lat is None or lon is None:
            print(f" -> Melewati baris {index}: Data tidak lengkap.")
            continue

        objek = None
        try:
            if 'Wisata' in tipe or tipe == 'Landmark':
                objek = TempatWisata(nama, lat, lon, tipe, deskripsi)

            elif tipe == 'Kuliner':
                objek = Kuliner(nama, lat, lon, deskripsi)

            elif 'Ibadah' in tipe:
                agama_info = "Umum"
                if "Islam" in tipe:   agama_info = "Islam"
                elif "Kristen" in tipe: agama_info = "Kristen"
                elif "Klenteng" in tipe: agama_info = "Tridharma"
                objek = TempatIbadah(nama, lat, lon, agama_info, deskripsi)

            # === TIPE BARU (Penugasan) ===
            elif tipe == 'Museum':
                objek = Museum(nama, lat, lon, "Arkeologi & Kebudayaan", deskripsi)

            elif tipe == 'Taman Kota':
                objek = TamanKota(nama, lat, lon, "Area hijau, jogging track", deskripsi)

            elif tipe == 'Kantor Pemerintahan':
                objek = KantorPemerintahan(nama, lat, lon, "Pemerintah Daerah", deskripsi)

            else:
                print(f" -> Peringatan: Tipe '{tipe}' untuk '{nama}' tidak dikenali.")

            if objek:
                list_objek_lokasi.append(objek)

        except Exception as e:
            print(f" -> GAGAL membuat objek untuk '{nama}' di baris {index}: {e}")

    print(f"Total {len(list_objek_lokasi)} objek lokasi berhasil dibuat "
          f"dari {len(dataframe)} baris data.")
    return list_objek_lokasi


# -------------------------------------------------------------------
# FUNGSI LOGGING
# -------------------------------------------------------------------

def tulis_log(pesan: str, file_log: str = "proses_peta.log"):
    """Menulis pesan log ke file dengan timestamp, menggunakan mode append."""
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    try:
        with open(file_log, 'a', encoding='utf-8') as f:
            f.write(f"[{timestamp}] {pesan}\n")
    except IOError as e:
        print(f"ERROR: Gagal menulis ke file log '{file_log}': {e}")


# -------------------------------------------------------------------
# FUNGSI BACA KONFIGURASI PETA (Penugasan)
# -------------------------------------------------------------------

def baca_config_peta(file_config: str = "config_peta.txt") -> tuple:
    """
    Membaca konfigurasi peta (latitude, longitude, zoom) dari file teks.
    Setiap nilai berada di baris terpisah.
    Jika file tidak ada atau terjadi error, mengembalikan nilai default Semarang.

    Returns:
        tuple: (latitude, longitude, zoom_start)
    """
    DEFAULT_LAT  = -6.9929
    DEFAULT_LON  = 110.4200
    DEFAULT_ZOOM = 13

    try:
        with open(file_config, 'r', encoding='utf-8') as f:
            baris = [line.strip() for line in f if line.strip()]

        lat  = float(baris[0])
        lon  = float(baris[1])
        zoom = int(baris[2])

        print(f" -> Konfigurasi peta berhasil dibaca dari '{file_config}':")
        print(f"    Latitude={lat}, Longitude={lon}, Zoom={zoom}")
        return (lat, lon, zoom)

    except FileNotFoundError:
        print(f" -> Peringatan: File '{file_config}' tidak ditemukan. "
              f"Menggunakan nilai default.")
    except (ValueError, IndexError) as e:
        print(f" -> Peringatan: Error membaca konfigurasi ({type(e).__name__}: {e}). "
              f"Menggunakan nilai default.")
    except Exception as e:
        print(f" -> Peringatan: Error tak terduga ({type(e).__name__}: {e}). "
              f"Menggunakan nilai default.")

    return (DEFAULT_LAT, DEFAULT_LON, DEFAULT_ZOOM)


# -------------------------------------------------------------------
# FUNGSI BUAT PETA (dengan kustomisasi marker + baca config + logging)
# -------------------------------------------------------------------

# Pemetaan tipe kelas → warna dan ikon Folium (Penugasan)
KONFIGURASI_MARKER = {
    TempatWisata:        {"color": "blue",      "icon": "camera",     "prefix": "fa"},
    Kuliner:             {"color": "red",        "icon": "cutlery",    "prefix": "fa"},
    TempatIbadah:        {"color": "purple",     "icon": "star",       "prefix": "fa"},
    Museum:              {"color": "orange",     "icon": "university", "prefix": "fa"},
    TamanKota:           {"color": "green",      "icon": "leaf",       "prefix": "fa"},
    KantorPemerintahan:  {"color": "cadetblue",  "icon": "building",   "prefix": "fa"},
}


def buat_peta_lokasi_folium(
    list_objek: list,
    file_output: str = "peta_lokasi.html",
    file_config: str = "config_peta.txt",
    file_log: str = "proses_peta.log"
):
    """
    Membuat peta Folium dari list objek Lokasi.
    - Membaca konfigurasi (lat, lon, zoom) dari file_config. (Penugasan)
    - Menggunakan ikon & warna berbeda per tipe objek via isinstance(). (Penugasan)
    - Menulis log proses ke file_log.
    """
    nama_fungsi = "buat_peta_lokasi_folium"

    if not list_objek:
        pesan = f"[{nama_fungsi}] Gagal: Tidak ada data lokasi untuk dipetakan."
        print(pesan)
        tulis_log(pesan, file_log)
        return

    # --- Baca konfigurasi peta dari file (Penugasan) ---
    print(f"\n[{nama_fungsi}] Membaca konfigurasi peta...")
    lat_tengah, lon_tengah, zoom_awal = baca_config_peta(file_config)

    tulis_log(
        f"[{nama_fungsi}] Memulai pembuatan peta '{file_output}' "
        f"dengan {len(list_objek)} lokasi. "
        f"Konfigurasi: lat={lat_tengah}, lon={lon_tengah}, zoom={zoom_awal}.",
        file_log
    )

    # --- Buat objek peta Folium ---
    peta = folium.Map(
        location=[lat_tengah, lon_tengah],
        zoom_start=zoom_awal,
        tiles="OpenStreetMap"
    )
    print(f"[{nama_fungsi}] Objek peta dibuat, berpusat di "
          f"({lat_tengah:.4f}, {lon_tengah:.4f}), zoom={zoom_awal}.")

    jumlah_marker   = 0
    lokasi_dilewati = []

    for lok in list_objek:
        koordinat = lok.get_koordinat()
        if koordinat == (0.0, 0.0):
            lokasi_dilewati.append(lok.nama)
            continue

        # --- Tentukan ikon & warna berdasarkan tipe objek (Penugasan) ---
        cfg_marker = KONFIGURASI_MARKER.get(type(lok), {
            "color": "gray", "icon": "info-sign", "prefix": "glyphicon"
        })

        ikon = folium.Icon(
            color=cfg_marker["color"],
            icon=cfg_marker["icon"],
            prefix=cfg_marker["prefix"]
        )

        folium.Marker(
            location=koordinat,
            popup=folium.Popup(lok.get_info_popup(), max_width=300),
            tooltip=f"{lok.nama} [{type(lok).__name__}]",
            icon=ikon
        ).add_to(peta)

        jumlah_marker += 1

    if lokasi_dilewati:
        pesan_lewat = (f"[{nama_fungsi}] Melewati marker untuk: "
                       f"{', '.join(lokasi_dilewati)} (koordinat tidak valid).")
        print(f" -> Peringatan: {pesan_lewat}")
        tulis_log(pesan_lewat, file_log)

    # --- Simpan peta ---
    try:
        peta.save(file_output)
        pesan_sukses = (f"[{nama_fungsi}] Peta '{file_output}' berhasil dibuat "
                        f"dengan {jumlah_marker} marker.")
        print(f"-> {pesan_sukses}")
        tulis_log(pesan_sukses, file_log)

    except Exception as e:
        pesan_error = (f"[{nama_fungsi}] ERROR saat menyimpan peta '{file_output}': "
                       f"{type(e).__name__} - {e}")
        print(f"-> {pesan_error}")
        tulis_log(pesan_error, file_log)


# -------------------------------------------------------------------
# KODE UTAMA
# -------------------------------------------------------------------

if __name__ == "__main__":
    NAMA_FILE_CSV    = "lokasi_semarang.csv"
    NAMA_FILE_PETA   = "peta_penugasan_semarang.html"
    FILE_CONFIG      = "config_peta.txt"
    FILE_LOG         = "proses_peta.log"

    print("=" * 60)
    print("  PENUGASAN JOBSHEET 12 — Mini Project SIG")
    print("  Nama : Muhammad Faqih Ramadhan | NIM : 4.33.25.0.17")
    print("=" * 60)

    # 1. Baca data CSV
    print("\n[1] Membaca data CSV...")
    df_lokasi = baca_data_lokasi(NAMA_FILE_CSV)

    # 2. Buat list objek lokasi
    print("\n[2] Membuat objek dari DataFrame...")
    list_semua_lokasi = buat_objek_lokasi_dari_df(df_lokasi)

    # 3. Tampilkan ringkasan objek per tipe
    print("\n[3] Ringkasan objek per tipe:")
    dari_tipe = {}
    for lok in list_semua_lokasi:
        tipe_nama = type(lok).__name__
        dari_tipe[tipe_nama] = dari_tipe.get(tipe_nama, 0) + 1
    for tipe_nama, jumlah in dari_tipe.items():
        print(f"    {tipe_nama:<25}: {jumlah} objek")

    # 4. Buat peta dengan kustomisasi marker & baca config
    print("\n[4] Membuat peta interaktif...")
    buat_peta_lokasi_folium(
        list_semua_lokasi,
        file_output=NAMA_FILE_PETA,
        file_config=FILE_CONFIG,
        file_log=FILE_LOG
    )

    print(f"\nSilakan buka '{NAMA_FILE_PETA}' di browser untuk melihat hasilnya.")
    print("\n--- Penugasan Selesai ---")


  PENUGASAN JOBSHEET 12 — Mini Project SIG
  Nama : Muhammad Faqih Ramadhan | NIM : 4.33.25.0.17

[1] Membaca data CSV...

[2] Membuat objek dari DataFrame...

Membuat objek dari DataFrame...
Total 16 objek lokasi berhasil dibuat dari 16 baris data.

[3] Ringkasan objek per tipe:
    TempatWisata             : 6 objek
    TempatIbadah             : 2 objek
    Kuliner                  : 2 objek
    Museum                   : 2 objek
    TamanKota                : 2 objek
    KantorPemerintahan       : 2 objek

[4] Membuat peta interaktif...

[buat_peta_lokasi_folium] Membaca konfigurasi peta...
 -> Peringatan: File 'config_peta.txt' tidak ditemukan. Menggunakan nilai default.
[buat_peta_lokasi_folium] Objek peta dibuat, berpusat di (-6.9929, 110.4200), zoom=13.
-> [buat_peta_lokasi_folium] Peta 'peta_penugasan_semarang.html' berhasil dibuat dengan 16 marker.

Silakan buka 'peta_penugasan_semarang.html' di browser untuk melihat hasilnya.

--- Penugasan Selesai ---
